# Spark Data Cleaning and Data Processing using DataFrames

## Objective
Understand Spark fundamentals and perform data cleaning, transformation, and aggregation using DataFrames

## Step 1: Import Required Libraries

Before working with the dataset, the necessary Spark libraries are imported. 

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

## Step 2: Load the Dataset

The dataset is stored as a CSV file. Spark reads the file and automatically identifies the column names and data types using the `header` and `inferSchema` options.

In [0]:
df = spark.read.csv("/Volumes/workspace/default/my_files/retail_sales.csv", header=True, inferSchema=True)
df.show(5)

+--------------+-------+----------------+---------------+--------------------+---+------------+------+---------+--------+----------------+-----------+-------+---------+-------------------+
|transaction_id|user_id|transaction_date|       username|               email|age|subscription|region|     city|store_id|product_category|sale_amount|  price|   status|      raw_timestamp|
+--------------+-------+----------------+---------------+--------------------+---+------------+------+---------+--------+----------------+-----------+-------+---------+-------------------+
|           130|   1160|      2026-01-01|  whitneyjustin|derrickhale@examp...| 48|       Basic| South|Hyderabad|       6|     Electronics|    3399.78| 389.57|     NULL|1982-12-26 19:49:53|
|           222|   1121|      2025-08-19|     jonathan70| scott37@example.org| 42|       Basic|  East|  Chennai|      20|           Books|    2110.99|1869.75|  Pending|1993-12-14 00:30:06|
|           227|   1055|      2026-03-29|      dpeterso

## Step 3: Explore the Dataset


In [0]:
df.printSchema()

df.show(10)

root
 |-- transaction_id: integer (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- username: string (nullable = true)
 |-- email: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- subscription: string (nullable = true)
 |-- region: string (nullable = true)
 |-- city: string (nullable = true)
 |-- store_id: integer (nullable = true)
 |-- product_category: string (nullable = true)
 |-- sale_amount: double (nullable = true)
 |-- price: double (nullable = true)
 |-- status: string (nullable = true)
 |-- raw_timestamp: timestamp (nullable = true)

+--------------+-------+----------------+---------------+--------------------+---+------------+------+---------+--------+----------------+-----------+-------+---------+-------------------+
|transaction_id|user_id|transaction_date|       username|               email|age|subscription|region|     city|store_id|product_category|sale_amount|  price|   status|      raw_timestam

## Step 4: Count Total Records


In [0]:
print("Total Records:", df.count())

Total Records: 800


## Step 5: Remove Duplicate Records
removing the duplicates using `dropDuplicates()` function.

In [0]:
df = df.dropDuplicates()
print("Records after removing duplicates:", df.count())

Records after removing duplicates: 800


From the Above we can say that there are no duplicates in out table 

## Step 6: Handle Missing Values
 The `status` column is also updated by replacing null values with the text **"Unknown"**.

In [0]:
df = df.na.drop(subset=["email"])
df = df.na.fill({"price":0,"status":"Unknown"})

## Step 7: Apply Filtering Conditions


Getting mumbai Users

In [0]:
mumbai_users = df.filter(col("city") == "Mumbai")
mumbai_users.show(5)

+--------------+-------+----------------+---------------+--------------------+---+------------+------+------+--------+----------------+-----------+-------+---------+-------------------+
|transaction_id|user_id|transaction_date|       username|               email|age|subscription|region|  city|store_id|product_category|sale_amount|  price|   status|      raw_timestamp|
+--------------+-------+----------------+---------------+--------------------+---+------------+------+------+--------+----------------+-----------+-------+---------+-------------------+
|           280|   1194|      2026-04-04|         kyle39| mmcneil@example.org| 55|     Premium|  East|Mumbai|       6|        Clothing|    4034.36| 578.44|  Pending|2013-03-13 11:25:42|
|            15|   1016|      2025-12-22|        tlozano|  ssmith@example.org| 24|     Premium|  West|Mumbai|      12|       Groceries|    1397.86|2815.73|  Unknown|1981-11-09 03:20:02|
|           274|   1018|      2026-05-13|santiagoshelley|robertbeck@ex

Applying and condition

In [0]:
mumbai_groceries = df.filter((col("city") == "Mumbai") & (col("product_category") == "Groceries"))
mumbai_groceries.show(5)

+--------------+-------+----------------+---------------+--------------------+---+------------+------+------+--------+----------------+-----------+-------+---------+-------------------+
|transaction_id|user_id|transaction_date|       username|               email|age|subscription|region|  city|store_id|product_category|sale_amount|  price|   status|      raw_timestamp|
+--------------+-------+----------------+---------------+--------------------+---+------------+------+------+--------+----------------+-----------+-------+---------+-------------------+
|            15|   1016|      2025-12-22|        tlozano|  ssmith@example.org| 24|     Premium|  West|Mumbai|      12|       Groceries|    1397.86|2815.73|  Unknown|1981-11-09 03:20:02|
|           274|   1018|      2026-05-13|santiagoshelley|robertbeck@exampl...| 34|       Basic|  East|Mumbai|       7|       Groceries|    2979.69|1507.32|Completed|2023-02-15 01:15:27|
|           251|   1014|      2026-04-17|     lanetracey| orogers@exam

## Step 8: Filter Sales from the West Region

To perform a simple business analysis, only sales from the **West** region are selected. The average sales amount is then calculated for each product category.

In [0]:
west_sales = df.filter(col("region") == "West")
grouped_sales = west_sales.groupBy("product_category")
final_analysis = grouped_sales.agg(avg("sale_amount").alias("Average Sales"))
final_analysis.show()

+----------------+------------------+
|product_category|     Average Sales|
+----------------+------------------+
|     Electronics|2764.4619607843133|
|       Groceries|2684.7312765957445|
|           Books| 2555.329649122807|
|        Clothing| 2587.388333333334|
+----------------+------------------+



## Step 9: Perform Aggregate Operations
 In this step, the total number of products, total sales, average price, minimum price, and maximum price are calculated.

In [0]:
#Counting total records
df.select(count("*")).show()

+--------+
|count(1)|
+--------+
|     800|
+--------+



In [0]:
#finding total price sum
df.select(sum("price").alias("Total Price")).show()

+------------------+
|       Total Price|
+------------------+
|1240874.8200000003|
+------------------+



In [0]:
#finding Average price
df.select(avg("price").alias("Average Price")).show()

+------------------+
|     Average Price|
+------------------+
|1551.0935250000005|
+------------------+



In [0]:
#Minimum and Maximum prices
df.select(min("price"),max("price")).show()

+----------+----------+
|min(price)|max(price)|
+----------+----------+
|     50.13|    2998.8|
+----------+----------+



## Step 10: Group Data by City

Grouping helps organize records based on a common column. Here, the total number of records is calculated for each city.

In [0]:
city_groups = df.groupBy("city")
city_groups.count().show()

+---------+-----+
|     city|count|
+---------+-----+
|  Chennai|  181|
|Hyderabad|  156|
|    Delhi|  157|
|   Mumbai|  163|
|Bangalore|  143|
+---------+-----+



## Step 11: Modify the DataFrame Schema

The `raw_timestamp` column is stored as a string in the dataset. It is converted into a proper Timestamp type and renamed as `event_time` for easier analysis.

In [0]:
#Converting raw_timestamp to a proper timestamp type
df = df.withColumn("event_time", col("raw_timestamp").cast(TimestampType()))

In [0]:
#Dropping the old column 
df = df.drop("raw_timestamp")

In [0]:
#new Schema
df.printSchema()

root
 |-- transaction_id: integer (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- username: string (nullable = true)
 |-- email: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- subscription: string (nullable = true)
 |-- region: string (nullable = true)
 |-- city: string (nullable = true)
 |-- store_id: integer (nullable = true)
 |-- product_category: string (nullable = true)
 |-- sale_amount: double (nullable = true)
 |-- price: double (nullable = false)
 |-- status: string (nullable = false)
 |-- event_time: timestamp (nullable = true)



## Step 12: Build a Complete Data Processing Pipeline

The final pipeline combines multiple data processing steps into a single workflow. It removes duplicate records, replaces missing prices with zero, groups data by store, and calculates the total revenue generated by each store.

In [0]:
#Dropping duplicates
clean_df = df.dropDuplicates()
#Handling missing prices
filled_df = clean_df.na.fill({"price": 0})
#Groupping by store
store_groups = filled_df.groupBy("store_id")
# findig each store revenue
final_df = store_groups.agg(sum("price").alias("Total Revenue"))
final_df.show()

+--------+------------------+
|store_id|     Total Revenue|
+--------+------------------+
|      10| 80568.06999999999|
|       7|          61125.78|
|      19|60678.799999999996|
|      14| 67780.18000000002|
|       3| 69029.55000000002|
|      11| 68414.20999999998|
|      15|          51317.86|
|       6|46551.350000000006|
|       1|          52237.93|
|      13|          93100.95|
|      16| 47506.71000000002|
|      12| 49591.93000000001|
|      20|          84159.53|
|       5|62287.130000000005|
|       2| 62343.56000000003|
|       8|51793.689999999995|
|      18|          56464.12|
|       4|          63270.66|
|       9|43932.149999999994|
|      17| 68720.65999999999|
+--------+------------------+



# Conclusion

This assignment demonstrated how Apache Spark DataFrames can be used to clean, transform, and analyze large datasets efficiently.

The following operations were successfully performed:

- Loaded a CSV dataset into Spark.
- Explored the dataset structure.
- Removed duplicate records.
- Handled missing values.
- Applied filtering conditions.
- Performed aggregation operations.
- Grouped data for analysis.
- Modified the DataFrame schema.
- Built a complete data processing pipeline.

These operations highlight the advantages of Spark's in-memory processing and immutable DataFrames for large-scale data analysis.